In [ ]:
import time
from dataclasses import dataclass

import pandas as pd
import torch
from linops import LinearOperator
from sklearn.datasets import load_svmlight_file
from sklearn.preprocessing import StandardScaler

from rlaopt.linalg import LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig, PCGState, PCGStoppingCriteria

In [ ]:
# torch.set_default_dtype(torch.float32)
torch.set_default_dtype(torch.float64)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

### Load data

In [ ]:
def standardize_and_convert_to_torch(X, y):
    # Standardize features and labels
    X_std = StandardScaler().fit_transform(X)
    y_std = StandardScaler().fit_transform(y.reshape(-1, 1)).flatten()

    X_torch = torch.tensor(X_std)
    y_torch = torch.tensor(y_std)

    return X_torch, y_torch


def load_data(file_path: str):
    X, y = load_svmlight_file(file_path)
    X_dense = X.toarray()

    X_torch, y_torch = standardize_and_convert_to_torch(X_dense, y)
    return X_torch, y_torch

In [ ]:
def gaussian_rand_features(X, n_features, bandwidth):
    W = (
        1
        / bandwidth
        * torch.randn((X.shape[1], n_features), device=X.device)
        / (n_features**0.5)
    )
    b = 2 * torch.pi * torch.rand((n_features,), device=X.device)
    return torch.cos(X @ W + b) * (2 / n_features) ** 0.5


def relu_rand_features(X, n_features):
    W = torch.randn((X.shape[1], n_features), device=X.device) / (n_features**0.5)
    return torch.relu(X @ W)

In [ ]:
# Taken from PROMISE paper (with more random features)
def load_acsincome():
    X = pd.read_pickle("./data/acsincome_data.pkl")
    y = pd.read_pickle("./data/acsincome_target.pkl")
    X_torch, y_torch = standardize_and_convert_to_torch(X.to_numpy(), y.to_numpy())
    X_torch = gaussian_rand_features(X_torch, 3000, bandwidth=1.0)
    return X_torch, y_torch


# Taken from GeNIOS paper
def load_e2006():
    X_torch, y_torch = load_data("./data/E2006.train.bz2")
    return X_torch, y_torch


# Taken from PROMISE paper
def load_realsim():
    X_torch, y_torch = load_data("./data/real-sim.bz2")
    return X_torch, y_torch


# Taken from PROMISE paper
def load_yearpredictionmsd():
    X_torch, y_torch = load_data("./data/YearPredictionMSD.bz2")
    X_torch = relu_rand_features(X_torch, X_torch.shape[0] // 100)
    return X_torch, y_torch

In [ ]:
X_torch, y_torch = load_yearpredictionmsd()

### Create linear system

In [ ]:
class GramOperator(LinearOperator):
    def __init__(self, X: torch.Tensor):
        super().__init__()

        self.register_buffer("_X", X)
        self._shape = (X.shape[1], X.shape[1])
        # self.device = X.device
        self.supports_operator_matrix = True

    def _matmul_impl(self, v: torch.Tensor) -> torch.Tensor:
        return self._X.T @ (self._X @ v)

    @property
    def device(self):
        return self._X.device

In [ ]:
def create_ridge_reg_linsys(X: torch.Tensor, y: torch.Tensor, reg: float) -> LinSys:
    A = GramOperator(X)
    b = X.T @ y
    return LinSys(A, b, reg)

In [ ]:
linsys = create_ridge_reg_linsys(X_torch, y_torch, reg=1e-2)
linsys = linsys.to(device)

### Create solver

In [ ]:
def create_pcg_solver(linsys):
    precond_config = NystromConfig(rank_init=100, base_damping=linsys.reg.item())
    return PCG(linsys, config=PCGConfig(preconditioner_config=precond_config))

In [ ]:
solver = create_pcg_solver(linsys)
num_iters = 5000

### Run solver

In [ ]:
@dataclass(kw_only=True, frozen=True)
class StepResult:
    residual_norm: float
    iteration: int
    time_elapsed: float


def step_result_from_state(state: PCGState, time_elapsed: float) -> StepResult:
    return StepResult(
        residual_norm=state.res_norm.item(),
        iteration=state.iter_,
        time_elapsed=time_elapsed,
    )


def print_step_result_with_frequency(step_result: StepResult, frequency: int = 20):
    if step_result.iteration % frequency == 0:
        print(
            f"Iter: {step_result.iteration:4d} | "
            f"Residual Norm: {step_result.residual_norm:.4e} | "
            f"Time Elapsed: {step_result.time_elapsed:.2f}s"
        )

In [ ]:
def run_solver(linsys, solver, num_iters):
    step_results = []

    params = torch.zeros_like(linsys.B)
    state = solver.init_state(params)

    step_results.append(step_result_from_state(state, 0.0))
    print_step_result_with_frequency(step_results[-1])

    for _ in range(num_iters):
        start_time = time.time()
        params, state = solver.step(params, state)
        time_elapsed = time.time() - start_time

        step_results.append(step_result_from_state(state, time_elapsed))
        print_step_result_with_frequency(step_results[-1])

    return params, step_results

In [ ]:
params, step_results = run_solver(linsys, solver, num_iters)

In [ ]:
result = solver.solve(
    stopping_criteria=PCGStoppingCriteria(max_iters=num_iters, tol=1e-6)
)

In [ ]:
result